In [48]:
import sys
sys.path.append("../modules")

from nlp import BERTWrapperForSA

from sklearn.metrics import f1_score, classification_report

import pandas as pd

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [39]:
bert_sa = BERTWrapperForSA("../models/finbert")

In [3]:
tweet_data = pd.read_parquet("../data/tweet_data.parquet")
tweets = tweet_data["text"]
tweet_data.head()

,id,created_at,text,hashtags,mentions,cashtags,is_reply,retweet_count,reply_count,like_count,quote_count,impression_count
0,1648218700875550721,2023-04-18T06:55:35.000Z,@Teslaconomics Intense work schedule,[],[Teslaconomics],[],True,1616,1810,35844,108,2067249
1,1648218093393420288,2023-04-18T06:53:10.000Z,@0xgaut 🔥🔥,[],[0xgaut],[],True,704,454,19838,26,1046453
2,1648212090660716544,2023-04-18T06:29:19.000Z,@dogeofficialceo 🤣,[],[dogeofficialceo],[],True,393,358,6986,24,632781
3,1648199231688134657,2023-04-18T05:38:13.000Z,@Not_the_Bee Sigh,[],[Not_the_Bee],[],True,1205,961,25641,59,1377332
4,1648198647283163136,2023-04-18T05:35:54.000Z,@IncompetentHum3 !,[],[IncompetentHum3],[],True,2959,1420,28808,109,1529066


In [4]:
n_tweets = 100
tweet_min_length = 50
selected_tweets = pd.DataFrame(tweets.loc[tweets.apply(lambda x: len(x) >= tweet_min_length)]).sample(n=n_tweets, random_state=42).reset_index().drop(columns=["index"])
# selected_tweets.to_csv("../data/test_tweets.csv")

In [40]:
test_data = pd.read_csv("../data/test_tweets_labeled.csv")
test_tweets = test_data["text"].tolist()
labels = test_data["label_0"].tolist()

In [41]:
greedy_preds = bert_sa(test_tweets)

In [62]:
bert_sa.amplification_threshold = 0.35
bert_sa.sentiment_dominance_ratio = 2
amplified_preds = bert_sa(test_tweets, decoding="amplified")

In [63]:
f1_score_greedy = f1_score(labels, greedy_preds, average="macro")
f1_score_amplified = f1_score(labels, amplified_preds, average="macro")
clf_report_greedy = classification_report(labels, greedy_preds)
clf_report_amplified = classification_report(labels, amplified_preds)

print("Macro F1-scores")
print("\tGreedy: ", f1_score_greedy)
print("\tAmplified Sentiments: ", f1_score_amplified)
print()
print("Classification Reports")
print("\t Greedy:\n", clf_report_greedy)
print("\t Amplified Sentiments:\n", clf_report_amplified)

Macro F1-scores
	Greedy:  0.5514492753623189
	Amplified Sentiments:  0.6297584541062802

Classification Reports
	 Greedy:
               precision    recall  f1-score   support

    negative       0.75      0.50      0.60         6
     neutral       0.61      0.96      0.75        56
    positive       0.88      0.18      0.30        38

    accuracy                           0.64       100
   macro avg       0.75      0.55      0.55       100
weighted avg       0.72      0.64      0.57       100

	 Amplified Sentiments:
               precision    recall  f1-score   support

    negative       0.67      0.67      0.67         6
     neutral       0.66      0.96      0.78        56
    positive       0.92      0.29      0.44        38

    accuracy                           0.69       100
   macro avg       0.75      0.64      0.63       100
weighted avg       0.76      0.69      0.65       100

